# Transformer Architecture: Encoder

Classify 8-token English sentences as grammatically valid (`ok`) or not (`bad`).

Each section adds one idea: flat MLP → attention on sets → attention on sequences (learned PE) → RoPE.

In [ ]:
import warnings
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from lark import Lark

warnings.filterwarnings("ignore", message="enable_nested_tensor")

## 1. Grammar

The vocabulary has 25 tokens across 7 categories. Valid sentences follow 8 sentence patterns.

- **Subject/object:** `article + entity_noun` (THE/A/AN agreement by initial sound)
- **Prepositional phrase:** `preposition + article + place_noun`
- **PP position:** beginning, end, or between subject and verb — nowhere else
- **Padding:** `<BLANK>` tokens must be contiguous at the end

In [ ]:
ENTITY_CONS  = ["CAT", "MOUSE", "ROBOT"]
ENTITY_VOWEL = ["ALLIGATOR", "ANT"]
PLACE_CONS   = ["HOUSE", "YARD", "STREET"]
PLACE_VOWEL  = ["AIRPORT", "OCEAN"]
INTRANS = ["SMILES", "BURPS", "SLEEPS", "EXPLODES"]
TRANS   = ["CHASES", "EATS", "LOVES", "HATES"]
PREPS   = ["IN", "UNDER", "ABOVE"]
BLANK   = "<BLANK>"

def alt(*words):
    return " | ".join(f'"{w}"' for w in words)

GRAMMAR = f"""
    sentence: subj intrans
            | subj trans obj
            | pp subj intrans
            | subj intrans pp
            | subj pp intrans
            | pp subj trans obj
            | subj trans obj pp
            | subj pp trans obj

    subj: "THE" entity | "A" entity_cons | "AN" entity_vowel
    obj:  "THE" entity | "A" entity_cons | "AN" entity_vowel
    pp:   prep "THE" place | prep "A" place_cons | prep "AN" place_vowel

    entity:       entity_cons | entity_vowel
    entity_cons:  {alt(*ENTITY_CONS)}
    entity_vowel: {alt(*ENTITY_VOWEL)}
    place:        place_cons | place_vowel
    place_cons:   {alt(*PLACE_CONS)}
    place_vowel:  {alt(*PLACE_VOWEL)}

    prep:   {alt(*PREPS)}
    intrans:{alt(*INTRANS)}
    trans:  {alt(*TRANS)}

    %ignore " "
"""

print(GRAMMAR)

In [ ]:
parser = Lark(GRAMMAR, start="sentence", parser="lalr")

def is_valid(tokens):
    """Check if an 8-token padded sequence is a valid sentence."""
    seen_blank = False
    for t in tokens:
        if t == BLANK:    seen_blank = True
        elif seen_blank:  return False        # non-BLANK after BLANK
    core = [t for t in tokens if t != BLANK]
    if not core: return False
    try:
        parser.parse(" ".join(core))
        return True
    except Exception:
        return False

examples = [
    ["THE", "CAT",       "SMILES", BLANK, BLANK, BLANK, BLANK, BLANK],  # ok
    ["IN",  "THE", "HOUSE", "THE", "CAT", "SMILES", BLANK, BLANK],      # ok: PP first
    ["THE", "CAT", "IN", "THE", "HOUSE", "SMILES", BLANK, BLANK],       # ok: PP middle
    ["THE", "CAT", "CHASES", "THE", "MOUSE", BLANK, BLANK, BLANK],      # ok: transitive
    ["AN",  "CAT",       "SMILES", BLANK, BLANK, BLANK, BLANK, BLANK],  # bad: AN + consonant
    ["THE", "HOUSE",     "SMILES", BLANK, BLANK, BLANK, BLANK, BLANK],  # bad: place noun as subject
    ["THE", "CAT",  BLANK, "SMILES", BLANK, BLANK, BLANK, BLANK],       # bad: BLANK in middle
]
for s in examples:
    print(f"{'ok ' if is_valid(s) else 'bad'} | {' '.join(s)}")

## 2. Dataset

The dataset was generated by:
1. Starting with valid sentences, applying 1–4 random token replacements or swaps
2. Labeling each result as `ok` or `bad`
3. Filling any valid shortfall with unmodified sentences

Result: exactly 50% ok, 50% bad — balanced binary classification.

In [ ]:
# Use 10k training rows — fast enough to re-run live during a lecture (~30s total).
# Full 1M-sample hyperparameter search is in scripts/mlp_search.py and encoder_search.py.
train_df    = pd.read_csv("data/train.csv",    nrows=10_000)
validate_df = pd.read_csv("data/validate.csv")
test_df     = pd.read_csv("data/test.csv")

print(f"{len(train_df):>9,} training rows (subset)")
print(f"{len(validate_df):>9,} validation rows")
print(f"{len(test_df):>9,} test rows")
print()
print(train_df["label"].value_counts())

In [ ]:
# Token vocabulary (same order as the dataset generator)
TOKENS = (["THE","A","AN"] + PREPS
          + ENTITY_CONS + ENTITY_VOWEL
          + PLACE_CONS  + PLACE_VOWEL
          + INTRANS + TRANS + [BLANK])
TOKEN_IDX  = {t: i for i, t in enumerate(TOKENS)}
VOCAB_SIZE = len(TOKENS)   # 25
SEQ_LEN    = 8

print(f"{VOCAB_SIZE} tokens × {SEQ_LEN} positions = {VOCAB_SIZE * SEQ_LEN}-dimensional one-hot")
print(TOKENS)

In [ ]:
SEQ_COLS = [f"t{i}" for i in range(SEQ_LEN)]

def df_to_indices(df):
    """Convert dataframe to (token_indices, labels) tensors."""
    idx = torch.tensor(
        df[SEQ_COLS].apply(lambda col: col.map(TOKEN_IDX)).values,
        dtype=torch.long,
    )
    y = torch.tensor((df["label"] == "ok").astype(int).values, dtype=torch.long)
    return idx, y

def indices_to_onehot(idx):
    """(n, 8) token indices → (n, 200) one-hot vectors."""
    n = idx.shape[0]
    x = torch.zeros(n, SEQ_LEN * VOCAB_SIZE)
    for pos in range(SEQ_LEN):
        x[torch.arange(n), pos * VOCAB_SIZE + idx[:, pos]] = 1.0
    return x

tr_idx, tr_y   = df_to_indices(train_df)
va_idx, va_y   = df_to_indices(validate_df)
te_idx, te_y   = df_to_indices(test_df)

tr_x = indices_to_onehot(tr_idx)
va_x = indices_to_onehot(va_idx)
te_x = indices_to_onehot(te_idx)

print(f"One-hot shape: {tr_x.shape}   Label shape: {tr_y.shape}")
print(f"Example: {train_df.iloc[0][SEQ_COLS].tolist()} → {train_df.iloc[0]['label']}")
print(f"         {tr_x[0].nonzero().flatten().tolist()}")

## 3. MLP Baseline

A plain **multi-layer perceptron** sees all 200 input dimensions simultaneously with no notion of sequence.
It can learn *which token appears at which position*, but only by memorizing position-specific patterns.

(Hyperparameters chosen by grid search over the full 1M-sample dataset.)

In [ ]:
mlp = nn.Sequential(
    nn.Linear(SEQ_LEN * VOCAB_SIZE, 128),
    nn.ReLU(),
    nn.Linear(128, 2),
)
print(f"Parameters: {sum(p.numel() for p in mlp.parameters()):,}")

In [ ]:
def train(model, tr_x, tr_y, va_x, va_y, lr=1e-3, batch_size=256, epochs=20, patience=4):
    opt     = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    loader  = DataLoader(TensorDataset(tr_x, tr_y), batch_size=batch_size, shuffle=True)
    train_losses, val_losses = [], []
    best_val, wait = float("inf"), 0

    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(xb)
        train_losses.append(total / len(tr_x))

        model.eval()
        with torch.no_grad():
            va_out = model(va_x)
            vl  = loss_fn(va_out, va_y).item()
            acc = (va_out.argmax(1) == va_y).float().mean().item()
        val_losses.append(vl)
        print(f"  epoch {epoch+1:2d}  train={train_losses[-1]:.4f}  val={vl:.4f}  acc={acc:.4f}")

        if vl < best_val - 1e-4:
            best_val, wait = vl, 0
        else:
            wait += 1
            if wait >= patience: break

    return train_losses, val_losses

mlp_train_loss, mlp_val_loss = train(mlp, tr_x, tr_y, va_x, va_y)

In [ ]:
def plot_losses(*runs, title=""):
    """runs: list of (label, train_losses, val_losses)"""
    fig, ax = plt.subplots(figsize=(10, 3))
    for label, tl, vl in runs:
        ax.plot(tl, label=f"{label} train")
        ax.plot(vl, label=f"{label} val", linestyle="--")
    ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy loss")
    if title: ax.set_title(title)
    ax.legend(); plt.tight_layout(); plt.show()

def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        return (model(x).argmax(1) == y).float().mean().item()

plot_losses(("MLP", mlp_train_loss, mlp_val_loss), title="MLP training")
print(f"Test accuracy: {accuracy(mlp, te_x, te_y):.4f}")

## 4. Encoder (Transformer)

An encoder maps each token to a vector, then lets tokens **attend to each other** via self-attention.

```
token indices  →  embedding  →  [self-attention + FFN] × N  →  mean pool  →  classifier
```

Three variants below differ only in how position information is (or isn't) provided.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, d_model=32, n_heads=2, n_layers=1, d_ff=128,
                 dropout=0.0, pos_enc="none"):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, d_model)
        self.pos_enc   = pos_enc
        if pos_enc == "learned":
            self.pos_emb = nn.Embedding(SEQ_LEN, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, 2)

    def forward(self, x):                     # x: (batch, seq) token indices
        emb = self.embedding(x)               # (batch, seq, d_model)
        if self.pos_enc == "learned":
            pos = torch.arange(x.size(1), device=x.device)
            emb = emb + self.pos_emb(pos)
        elif self.pos_enc == "rope":
            emb = rope_encode(emb)
        out = self.transformer(emb)           # (batch, seq, d_model)
        return self.head(out.mean(dim=1))     # (batch, 2)

### 4a. No Position Encoding

Without position information, attention operates on a **set** of tokens — order is invisible.

The model can still learn bag-of-words rules: "must contain exactly one verb", "must contain an article", etc.
But it cannot learn order-dependent rules: "article must come before noun", "PP must not split verb from object".

Expected: accuracy much worse than the MLP.

In [ ]:
enc_none = Encoder(n_layers=1, pos_enc="none")
print(f"Parameters: {sum(p.numel() for p in enc_none.parameters()):,}")

none_train_loss, none_val_loss = train(enc_none, tr_idx, tr_y, va_idx, va_y)
plot_losses(("no-PE encoder", none_train_loss, none_val_loss), title="Encoder (no position encoding)")
print(f"Test accuracy: {accuracy(enc_none, te_idx, te_y):.4f}")

In [ ]:
# Demonstrate set behavior: permuting tokens doesn't change the prediction.
import random; random.seed(0)

sample = train_df[train_df["label"] == "ok"].iloc[0][SEQ_COLS].tolist()
print(f"Original:  {sample}")

shuffled = sample[:]; random.shuffle(shuffled)
print(f"Shuffled:  {shuffled}")

def predict_proba(model, tokens):
    idx = torch.tensor([[TOKEN_IDX[t] for t in tokens]])
    with torch.no_grad():
        logits = model(idx)
    probs = logits.softmax(-1)[0]
    return f"p(ok)={probs[1]:.3f}"

print(f"  Original prediction: {predict_proba(enc_none, sample)}")
print(f"  Shuffled prediction: {predict_proba(enc_none, shuffled)}")
print()

# Show: model responds to verb count
VERB = INTRANS + TRANS
def verb_count_demo(model):
    base = ["THE", "CAT", BLANK, BLANK, BLANK, BLANK, BLANK, BLANK]
    for n_verbs in range(4):
        tokens = ["THE", "CAT"] + INTRANS[:n_verbs] + [BLANK]*(6 - n_verbs)
        print(f"  {n_verbs} verb(s): {tokens[:4]}...  {predict_proba(model, tokens)}")

print("No-PE encoder response to verb count:")
verb_count_demo(enc_none)

### 4b. Learned Position Encoding

Add a learnable vector for each of the 8 positions and add it to the token embedding:

```python
emb = token_embedding(x) + position_embedding(positions)
```

Now the model *can* distinguish position 0 from position 7.

**Why fewer parameters than the MLP?**  
The MLP uses a 200-dimensional one-hot input (one slot per token × position), learning separate weights for each `(position, token)` pair.  
The encoder uses a shared 25-token vocabulary (not position-specific), then adds a small 8-position embedding — far more parameter-efficient.

In [ ]:
enc_learned = Encoder(n_layers=1, pos_enc="learned")
print(f"Parameters: {sum(p.numel() for p in enc_learned.parameters()):,}")

learned_train_loss, learned_val_loss = train(enc_learned, tr_idx, tr_y, va_idx, va_y)
plot_losses(("learned-PE encoder", learned_train_loss, learned_val_loss),
            title="Encoder (learned position encoding)")
print(f"Test accuracy: {accuracy(enc_learned, te_idx, te_y):.4f}")

In [ ]:
# The encoder achieves the same accuracy as the MLP with ~half the parameters,
# because it shares token representations across positions instead of encoding
# each position separately in the 200-dim one-hot input.
print("Parameter comparison:")
print(f"  MLP (200 → 128 → 2):    {sum(p.numel() for p in mlp.parameters()):>6,}  (position-specific one-hot)")
print(f"  Encoder (no PE):         {sum(p.numel() for p in enc_none.parameters()):>6,}  (token embeddings, no positions)")
print(f"  Encoder (learned PE):    {sum(p.numel() for p in enc_learned.parameters()):>6,}  (token + position embeddings)")
print()
print("Accuracy comparison:")
print(f"  MLP:                     {accuracy(mlp,         te_x,   te_y):.4f}")
print(f"  Encoder (no PE):         {accuracy(enc_none,    te_idx, te_y):.4f}  ← position-blind: can only use bag-of-words")
print(f"  Encoder (learned PE):    {accuracy(enc_learned, te_idx, te_y):.4f}  ← same accuracy, half the parameters")

### 4c. RoPE (Rotary Position Encoding)

**Idea:** instead of adding a fixed positional vector, *rotate* the query and key vectors by an angle proportional to position.

For a 2D vector `(x, y)` at position `p`, RoPE applies:

```
(x, y) → (x·cos(pθ) − y·sin(pθ),  x·sin(pθ) + y·cos(pθ))
```

For higher dimensions, apply this rotation independently to each successive pair of dimensions, each with a different frequency θ.

Key property: the dot product `Q·K` depends only on the **relative** position `p_q − p_k` — not on absolute positions. This generalizes well to sequence lengths longer than those seen in training.

In [ ]:
import math

def rope_encode(emb):
    """Apply RoPE rotation to token embeddings.
    emb: (batch, seq, d_model)  —  d_model must be even
    """
    batch, seq, d = emb.shape
    half = d // 2
    # One frequency per dimension-pair: θ_i = 1 / 10000^(2i/d)
    theta = 1.0 / (10000 ** (torch.arange(half, device=emb.device).float() / half))
    # Rotation angles: (seq, half)
    angles = torch.outer(torch.arange(seq, device=emb.device).float(), theta)
    cos = angles.cos()
    sin = angles.sin()
    x1, x2 = emb[..., :half], emb[..., half:]
    # Rotate each pair: [x1, x2] → [x1·cos − x2·sin, x1·sin + x2·cos]
    return torch.cat([x1 * cos - x2 * sin,
                      x1 * sin + x2 * cos], dim=-1)


# ── Visualization ─────────────────────────────────────────────────────────────
# In 2D: a single pair with θ = 1.0.  Each position rotates by 1 radian.
theta_2d = 1.0   # = 1 / 10000^(0/1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect("equal")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)

for pos in range(8):
    angle = pos * theta_2d
    rx =  math.cos(angle)   # "THE" starts at (1, 0)
    ry =  math.sin(angle)
    ax.annotate("", xy=(rx, ry), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=plt.cm.viridis(pos/7), lw=2))
    ax.text(rx * 1.13, ry * 1.13, f"pos {pos}", ha="center", va="center", fontsize=9)

ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
ax.set_title('RoPE rotation of "THE" embedding at each position\n(2D projection, θ = 1.0 rad/position)')
plt.tight_layout(); plt.show()

In [ ]:
enc_rope = Encoder(n_layers=1, pos_enc="rope")
print(f"Parameters: {sum(p.numel() for p in enc_rope.parameters()):,}")

rope_train_loss, rope_val_loss = train(enc_rope, tr_idx, tr_y, va_idx, va_y)
plot_losses(("RoPE encoder", rope_train_loss, rope_val_loss), title="Encoder (RoPE)")
print(f"Test accuracy: {accuracy(enc_rope, te_idx, te_y):.4f}")

In [ ]:
print("Final comparison:")
print(f"{'Model':<25} {'Params':>8}  {'Test acc':>9}")
print("-" * 46)
for name, model, x, y in [
    ("MLP (1×128)",          mlp,         te_x,   te_y),
    ("Encoder (no PE)",      enc_none,    te_idx, te_y),
    ("Encoder (learned PE)", enc_learned, te_idx, te_y),
    ("Encoder (RoPE)",       enc_rope,    te_idx, te_y),
]:
    n = sum(p.numel() for p in model.parameters())
    print(f"  {name:<23} {n:>8,}  {accuracy(model, x, y):>9.4f}")

plot_losses(
    ("no PE",      none_train_loss,    none_val_loss),
    ("learned PE", learned_train_loss, learned_val_loss),
    ("RoPE",       rope_train_loss,    rope_val_loss),
    title="Encoder variants — validation loss",
)

## 5. How parameter count scales with sequence length

Let $L$ = sequence length, $V$ = vocabulary size, $H$ = MLP hidden width, $d$ = encoder d_model.

---

### MLP

The input is a concatenated one-hot vector of dimension $LV$.  The first layer maps $LV \to H$:

$$N_\text{MLP} = \underbrace{LVH}_{\text{layer 1}} + \underbrace{2H}_{\text{layer 2}} = H(LV + 2)$$

The dominant cost is **layer 1**, which must learn a separate weight for every (position, token) pair.
Adding one token position costs exactly $VH$ new parameters.

$$N_\text{MLP} = \Theta(L)$$

with constant $VH$.  If representational capacity demands $H \propto L$ (more positions → more patterns to memorize), the cost becomes $\Theta(L^2)$.

---

### Encoder

The attention and feed-forward blocks apply the **same weight matrices** to every token position.  For one transformer layer with feed-forward width $d_{ff}$:

$$N_\text{layer} = \underbrace{4d^2 + 4d}_{\text{self-attention}} + \underbrace{2d\,d_{ff} + d_{ff} + d}_{\text{FFN}} + \underbrace{4d}_{\text{layer norms}}$$

None of these depend on $L$.  The only $L$-dependent term is the positional encoding:

| Encoding | $L$-dependent params | 
|---|---|
| Learned PE | $Ld$ |
| RoPE | $0$ (rotation computed on-the-fly) |

So the full parameter count is:

$$N_\text{enc} = \underbrace{Vd}_{\text{token emb.}} + \underbrace{Ld}_{\text{pos. emb. (learned)}} + \underbrace{n_\text{layers}(4d^2 + 2d\,d_{ff} + \cdots)}_{\text{position-independent}} + \underbrace{2d}_{\text{head}}$$

$$N_\text{enc} = \Theta(1) \quad \text{(RoPE)} \qquad N_\text{enc} = \Theta(L) \quad \text{(learned PE)}$$

Both are $O(L)$, but the learned PE constant is $d$, vs $VH$ for the MLP.

---

### Comparing the constants

In this notebook: $V = 25$, $H = 128$, $d = 32$.

| | params added per new position |
|---|---|
| MLP | $VH = 25 \times 128 = 3{,}200$ |
| Encoder, learned PE | $d = 32$ |
| Encoder, RoPE | $0$ |

The MLP adds **100× more parameters per position** than learned PE.  At $L = 8$ that difference accounts for nearly the entire gap (25,986 vs 13,826).

**Why?** The MLP has no shared representation across positions: it encodes "token $t$ at position $p$" as a distinct input dimension, so each position contributes a fresh $V \times H$ block to the weight matrix.  The encoder embeds token identity once (a shared $V \times d$ table) and then lets attention combine positions dynamically — the $L$-scaled cost collapses to a single lookup per position ($d$ parameters for learned PE, zero for RoPE).